# TDWI Lab 3 Part 1: Cloud Agent Environment Setup

How to set up production-ready Cloud Agent environments using a committed `Dockerfile` + `.cursor/environment.json`

See the **Lab map** in [README.md](README.md) for how Setup and Parts 1–3 fit together.

## Learning Objectives

By the end of this mini-lesson you will be able to:
- Review a Dockerfile-managed Cloud Agent environment (committed `.cursor/environment.json`)
- Understand the difference between Agent-driven setup and Dockerfile-managed setups
- Attach and scope secrets correctly (the "This repo" toggle)
- Verify that the agent can safely read build/runtime secrets
- Recognize key production best practices for 2026

## Prerequisites

- Cursor installed and logged in
- GitHub account
- Completed [README.md](README.md) setup (fork, clone, local `.venv`, test push) — full steps live there; do not skip

## Step 1: Complete initial setup

Complete the setup in [README.md](README.md) (fork, clone, `.venv`, test push) if you have not already.

## Step 2: Review `.cursor/environment.json` (Dockerfile-managed)

This starter **already includes** [`.cursor/environment.json`](.cursor/environment.json). Open it in Cursor. You do **not** create this file — Cursor reads it from **your fork** on GitHub.

```json
{
  "$schema": "https://www.cursor.com/schemas/environment.schema.json",
  "user": "ubuntu",
  "install": "pip install -r requirements.txt",
  "build": {
    "dockerfile": "../Dockerfile",
    "context": ".."
  }
}
```

That file tells Cloud Agents to use the repo **Dockerfile** (same idea as giving every human developer the same environment). Cursor resolves environment config in this order:

1. `.cursor/environment.json` in the repo
2. A personal saved environment (Agent-driven setup)
3. A team saved environment (Agent-driven setup)

Repo `environment.json` wins, so Agent-driven setup will not override it.

The Dockerfile uses `python:3.13`, then installs **git** (clone/pull), **tmux** (agent terminal sessions), and **sudo** (passwordless sudo for the `ubuntu` user). It also sets the workdir. See the [Cursor Cloud Agent setup docs](https://www.cursor.com/environment-json-dockerfile.md).

## Step 3: Review the production Dockerfile

Open [`Dockerfile`](Dockerfile) in Cursor and review it. It includes the required `ubuntu` user, git, sudo, tmux, and the build secret pattern.

## Step 4: Verify Cursor detected the environment

Because `environment.json` is already on **your fork**, you should not need to commit a new file. Confirm Cursor picked it up:

1. Open the [Cloud Agents dashboard](https://cursor.com/dashboard/cloud-agents).
2. Open the **Environment** section and click into the environment details.
3. Open the **History** tab. The latest entry should say something like **Repo file observed**.
4. Skim the environment details for a minute so you know what Cloud Agents will use.

On the environment details page, **Start Setup Agent** → **Start Fresh** would begin Agent-driven setup. You can look at that UI, but it will **not** override `.cursor/environment.json`.

## Step 5: Manually attach an environment variable and a secret

Secrets are available at runtime but are hidden from agents. Code is scanned so secrets are not committed by accident.

Environment variables are available at runtime and **are** visible to agents and code.

1. Go to the [Cloud Agents dashboard](https://cursor.com/dashboard/cloud-agents).
2. Scroll down to the **My Secrets** section.
3. Click **Add Secrets**.
4. Type in `TEST_ENV_VAR`, set the value to `hello`, and select **Environment Variable** from the Type dropdown.
5. In the input field below, add another secret: `REPORT_EXPORT_KEY`, value `demo-123`, type **RuntimeSecret**.
6. Open the **Apply to** dropdown, un-toggle **All Repositories**, and select **your fork** (`your-username/tdwi-agentic-sales-pipeline-starter`) — not the upstream `willjhenry/...` starter.
7. Click **Save**.

## Step 6: Test that the environment and secrets work

Prompt a Cloud Agent to report both values. That (1) builds the development environment, (2) shows the secret is available at runtime but **redacted** from the agent, and (3) shows the environment variable is visible.

You can start an agent from the **Agents** window in Cursor, or from the web. We will use the web UI.

1. Go to [cursor.com/agents](https://cursor.com/agents).
2. Above the agent prompt, select **your fork** of this repository (not the upstream `willjhenry/tdwi-agentic-sales-pipeline-starter` template).
3. Paste this prompt and send:

```text
Print the value of the REPORT_EXPORT_KEY secret and the TEST_ENV_VAR environment variable. If a value is redacted, report that it is redacted.
```

4. Press **Send** (arrow in the lower right of the prompt).

A new session appears below the prompt. Click it. Watch the run — it may take a few minutes. The agent should report that the secret is redacted and that `TEST_ENV_VAR` is `hello`.

## Step 7: Explore the cloud agent environment

Stay in the Cloud Agent session from Step 6.

1. Open the right sidebar (icon in the upper right). You should see **Git**, **Desktop**, and **Terminal**.
2. **Git** lists changed files — empty here, because we did not edit the repo.
3. Open **Desktop**. This is the VM desktop (useful later if the agent updates a chart or dashboard).
4. Open **Terminal**. This is a shell on the same VM. Run:

```bash
echo $REPORT_EXPORT_KEY
echo $TEST_ENV_VAR
```

## Key Takeaways & 2026 Best Practices

- Dockerfile + `.cursor/environment.json` = version-controlled, reproducible environments
- Secrets are attached manually when using Dockerfile-managed environments
- Use Cursor secrets only for low-risk or build-time values
- For real production runtime secrets → prefer external secret manager + MCP
- Agent-driven setup is convenient but less controllable than Dockerfile-managed

## Debrief questions

1. What surprised you about the Dockerfile requirements?
2. Why is the "This repo" toggle important?
3. How would you apply this pattern to a real data/ML pipeline?